In [28]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# Load only the sample data; cp1252 handles the German characters in this file.
events = pd.read_csv("../../data/otto_sample_data.csv", encoding="cp1252")
events["Product_Price"] = pd.to_numeric(events["Product_Price"], errors="coerce")

# One catalog row per product, used only to display recommendation details.
product_columns = [
    "Product_ID", "Product_Name", "Department", "Product_Category",
    "Product_Sub_Category", "Brand_Name", "Product_Type", "Product_Price",
]
catalog = (events.dropna(subset=["Product_ID"])
           .sort_values("Event_Date")
           .drop_duplicates("Product_ID", keep="last")[product_columns]
           .reset_index(drop=True))

In [29]:
# Convert events into weighted implicit-feedback signals.
action_weights = {
    "Bought": 5, "Added_to_Cart": 3, "Wishlisted": 2,
    "Reviewed": 2, "Browsed": 1, "Exchanged": 1, "Returned": -2,
}
events["interaction_weight"] = events["Action"].map(action_weights).fillna(0)
events.loc[events["Rating"].notna(), "interaction_weight"] += events.loc[events["Rating"].notna(), "Rating"] - 3

# Rows are customers, columns are products, and values are aggregated positive signals.
interactions = (events.groupby(["Customer_ID", "Product_ID"], as_index=False)["interaction_weight"].sum())

interactions["interaction_weight"] = interactions["interaction_weight"].clip(lower=0)
user_item_matrix = interactions.pivot(
    index="Customer_ID", columns="Product_ID", values="interaction_weight"
).fillna(0)
product_ids = user_item_matrix.columns.to_numpy()
product_similarity = cosine_similarity(user_item_matrix.T)
product_popularity = user_item_matrix.sum(axis=0)




In [32]:
def recommend_for_customer(customer_id, top_n=10):
    if customer_id not in user_item_matrix.index:
        scores = product_popularity.copy()
    else:
        customer_vector = user_item_matrix.loc[customer_id].to_numpy()
        scores = customer_vector @ product_similarity
        scores = pd.Series(scores, index=product_ids)
        scores.loc[customer_vector > 0] = -np.inf

    result = (pd.DataFrame({"Product_ID": product_ids, "score": scores})
              .replace([np.inf, -np.inf], np.nan)
              .dropna(subset=["score"]))
    score_min, score_max = result["score"].min(), result["score"].max()
    result["score"] = 1.0 if score_min == score_max else (result["score"] - score_min) / (score_max - score_min)

    return (result.merge(catalog, on="Product_ID", how="left")
            .sort_values("score", ascending=False)
            .head(top_n)
            .reset_index(drop=True))

sample_customer = user_item_matrix.index[12]
recommendations = recommend_for_customer(sample_customer)
print(recommendations[["Product_ID", "Product_Name", "score"]])

  Product_ID                       Product_Name     score
0     P11002            Sporty Beige Maxi Dress  1.000000
1     P11062         Scandinavian Black OLED TV  0.938542
2     P11026                Modern Black Blazer  0.879057
3     P11020         Classic Black Skinny Jeans  0.819661
4     P11038     Classic White Fitted Sheet Set  0.791829
5     P11110      Minimalist Blue Mountain Bike  0.787375
6     P11047             Modern Beige Sectional  0.769854
7     P11060         Modern Black Pendant Light  0.755667
8     P11036  Minimalist Terracotta Duvet Cover  0.738764
9     P11077              Sporty Silver Monitor  0.733922


## Export Artifacts for FastAPI

Save every artifact required to generate collaborative-filtering recommendations without retraining.

In [33]:
from pathlib import Path
import joblib

model_dir = Path("../models")
model_dir.mkdir(exist_ok=True)
collaborative_filtering_artifacts = {
    "catalog": catalog,
    "user_item_matrix": user_item_matrix,
    "product_ids": product_ids,
    "product_similarity": product_similarity,
    "product_popularity": product_popularity,
}
artifact_path = model_dir / "collaborative_filtering.joblib"
joblib.dump(collaborative_filtering_artifacts, artifact_path)
print(f"Saved FastAPI artifacts to {artifact_path.resolve()}")

Saved FastAPI artifacts to C:\Users\DRUPAKUL\OneDrive - Otto Group\Documents\GitHub\Own Repo\Recommendation_Engine\app\models\collaborative_filtering.joblib
